# Comprehensive Results Analysis
## Bike Demand Forecasting Study

In [ ]:
#TODO: Update modles 
MODEL_ORDER = [
    "TabPFN",
    "SARIMAX",
    "TabPFN_NoWeather",
    "Seasonal_Naive",
    "ARIMA",
    "XGBoost",
]

In [180]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import helper function
import sys
sys.path.append('..')
from Archive.forecasting_visualization import filter_results

In [181]:
# Configure plotting style
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 8

# Colorblind-friendly palette
colors = sns.color_palette('Set2', 8)

In [182]:
# Create output directories
Path('../results/figures').mkdir(parents=True, exist_ok=True)
Path('../results/tables').mkdir(parents=True, exist_ok=True)

## Section 1: Load Data

In [183]:
# Load aggregated results
df_agg = pd.read_csv('../results/results_master_v3.csv')
print(f"Aggregated results shape: {df_agg.shape}")
print(f"Columns: {df_agg.columns.tolist()}")
print(f"\nUnique models: {df_agg['model'].unique()}")
print(f"Unique horizons: {sorted(df_agg['horizon'].unique())}")
print(f"Unique weather scenarios: {df_agg['weather_scenario'].unique()}")

Aggregated results shape: (44, 21)
Columns: ['dataset', 'run_name', 'timestamp', 'model', 'horizon', 'weather_scenario', 'model_uses_covariates', 'degradation_seed', 'num_weather_vars', 'n_folds', 'MAE_mean', 'MAE_std', 'RMSE_mean', 'RMSE_std', 'MASE_mean', 'MASE_std', 'sMAPE_mean', 'sMAPE_std', 'total_test_imputed', 'total_train_imputed', 'folds_with_imputed_test']

Unique models: ['Seasonal_Naive' 'ARIMA' 'SARIMAX' 'XGBoost' 'TabPFN' 'TabPFN_NoWeather']
Unique horizons: [np.int64(6), np.int64(24), np.int64(48), np.int64(168)]
Unique weather scenarios: ['clean_only' 'degraded']


In [184]:
# Load detailed fold-level results
df_detailed = pd.read_csv('../results/detailed_results_master_v3.csv')
print(f"\nDetailed results shape: {df_detailed.shape}")
print(f"Unique folds: {sorted(df_detailed['fold'].unique())}")


Detailed results shape: (880, 16)
Unique folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


In [185]:
# Filter to latest run
df_latest = filter_results(df_agg, dataset=None, run_name=None)
df_detailed_latest = filter_results(df_detailed, dataset=None, run_name=None)

In [186]:
# Create scenario subsets
df_clean = df_latest[df_latest['weather_scenario'] == 'clean_only'].copy()
df_degraded = df_latest[df_latest['weather_scenario'] == 'degraded'].copy()

df_detailed_clean = df_detailed_latest[df_detailed_latest['weather_scenario'] == 'clean_only'].copy()
df_detailed_degraded = df_detailed_latest[df_detailed_latest['weather_scenario'] == 'degraded'].copy()

print(f"Clean scenario: {len(df_clean)} rows")
print(f"Degraded scenario: {len(df_degraded)} rows")

Clean scenario: 24 rows
Degraded scenario: 12 rows


## Section 2: RQ1.1 Analysis - Foundation vs Traditional Models

In [188]:
df_clean.columns

Index(['dataset', 'run_name', 'timestamp', 'model', 'horizon',
       'weather_scenario', 'model_uses_covariates', 'degradation_seed',
       'num_weather_vars', 'n_folds', 'MAE_mean', 'MAE_std', 'RMSE_mean',
       'RMSE_std', 'MASE_mean', 'MASE_std', 'sMAPE_mean', 'sMAPE_std',
       'total_test_imputed', 'total_train_imputed', 'folds_with_imputed_test'],
      dtype='object')

In [189]:
# Overall model rankings by MAE (averaged across all horizons)
overall_rankings = df_clean.groupby('model').agg({
    'MAE_mean': 'mean',
    'RMSE_mean': 'mean',
    'MASE_mean': 'mean',
    'sMAPE_mean': 'mean'

}).round(2).sort_values('MAE_mean')

overall_rankings['Rank'] = range(1, len(overall_rankings) + 1)
overall_rankings = overall_rankings[['Rank', 'MAE_mean', 'RMSE_mean', 'MASE_mean', 'sMAPE_mean']]
overall_rankings.columns = ['Rank', 'MAE', 'RMSE', 'MASE', 'sMAPE']



print("Overall Model Rankings (Clean Weather Only):")
print(overall_rankings)

Overall Model Rankings (Clean Weather Only):
                  Rank     MAE     RMSE  MASE  sMAPE
model                                               
TabPFN               1  181.43   251.20  0.91  22.68
SARIMAX              2  304.32   385.96  1.52  36.24
TabPFN_NoWeather     3  330.99   453.61  1.67  38.31
Seasonal_Naive       4  396.97   544.27  2.00  43.83
ARIMA                5  728.29   856.65  3.65  70.64
XGBoost              6  880.20  1010.27  4.36  70.88


In [190]:
# Performance by horizon for all models
perf_by_horizon = df_clean.pivot_table(
    index='horizon',
    columns='model',
    values='MAE_mean'
).round(2)
# reorder columns according to MODEL_ORDER (keep only those present)
ordered_cols = [m for m in MODEL_ORDER if m in perf_by_horizon.columns]
perf_by_horizon = perf_by_horizon[ordered_cols]
# save CSV
perf_by_horizon.to_csv("../results/tables/table6_mae_by_horizon.csv")

# save LaTeX
latex = "\\begin{table}[ht!]\n"
latex += "\\centering\n"
latex += "\\begin{tabular}{r" + "r" * len(perf_by_horizon.columns) + "}\n"
latex += "\\toprule\n"

# header
latex += "Horizon"
for model in perf_by_horizon.columns:
    latex += f" & {model}"
latex += " \\\\\n"
latex += "\\midrule\n"

# rows
for horizon, row in perf_by_horizon.iterrows():
    latex += f"{int(horizon)}"
    for v in row:
        latex += f" & {v:.2f}"
    latex += " \\\\\n"

latex += "\\bottomrule\n"
latex += "\\end{tabular}\n"
latex += (
    "\\caption{Mean absolute error (MAE) by forecast horizon for all models "
    "(clean weather).}\n"
)
latex += "\\label{tab:mae_by_horizon}\n"
latex += "\\end{table}\n"

with open("../results/tables/table6_mae_by_horizon.tex", "w") as f:
    f.write(latex)
print("table6 saved")
print("\nMAE by Horizon:")
print(perf_by_horizon)

table6 saved

MAE by Horizon:
model    TabPFN  SARIMAX  TabPFN_NoWeather  Seasonal_Naive   ARIMA  XGBoost
horizon                                                                    
6        170.50   283.45            441.42          506.42  593.02   524.74
24       169.57   284.68            266.10          318.68  732.26   814.87
48       208.94   320.16            298.72          381.70  804.70  1141.79
168      176.70   329.01            317.73          381.07  783.18  1039.41


In [191]:
# Statistical comparison: TabPFN vs baselines
tabpfn_mae = df_clean[df_clean['model'] == 'TabPFN']['MAE_mean'].mean()
baseline_comparisons = []

for model in df_clean['model'].unique():
    if model != 'TabPFN':
        model_mae = df_clean[df_clean['model'] == model]['MAE_mean'].mean()
        diff = tabpfn_mae - model_mae
        pct_diff = (diff / model_mae) * 100
        baseline_comparisons.append({
            'Baseline': model,
            'Baseline_MAE': round(model_mae, 2),
            'TabPFN_MAE': round(tabpfn_mae, 2),
            'Difference': round(diff, 2),
            'Pct_Difference': round(pct_diff, 2)
        })

comparison_df = pd.DataFrame(baseline_comparisons).sort_values('Baseline_MAE')
print("\nTabPFN vs Baselines:")
print(comparison_df)


TabPFN vs Baselines:
           Baseline  Baseline_MAE  TabPFN_MAE  Difference  Pct_Difference
2           SARIMAX        304.32      181.43     -122.90          -40.38
3  TabPFN_NoWeather        330.99      181.43     -149.56          -45.19
0    Seasonal_Naive        396.97      181.43     -215.54          -54.30
1             ARIMA        728.29      181.43     -546.86          -75.09
4           XGBoost        880.20      181.43     -698.78          -79.39


In [194]:
# ==============================
# OVERWRITE TABLE 1 (MAE-ranked, deltas for ALL metrics)
# LaTeX uses \( \) instead of $ $
# ==============================

metrics = ["MAE_mean", "RMSE_mean", "MASE_mean", "sMAPE_mean"]

table1 = (
    df_clean.groupby("model")[metrics]
    .mean()
    .round(2)
    .sort_values("MAE_mean")
)

table1["Rank"] = range(1, len(table1) + 1)

table1 = table1.rename(columns={
    "MAE_mean": "MAE",
    "RMSE_mean": "RMSE",
    "MASE_mean": "MASE",
    "sMAPE_mean": "sMAPE",
})

best = table1.iloc[0]
for m in ["MAE", "RMSE", "MASE", "sMAPE"]:
    table1[f"Delta_{m}"] = (table1[m] - best[m]).round(2)

table1.to_csv("../results/tables/table1_overall_rankings.csv")

latex = "\\begin{table}[ht!]\n"
latex += "\\centering\n"
latex += "\\begin{tabular}{clrrrrrrrr}\n"
latex += "\\toprule\n"
latex += (
    "Rank & Model & MAE & \\(\\Delta\\) MAE & RMSE & \\(\\Delta\\) RMSE & "
    "MASE & \\(\\Delta\\) MASE & sMAPE & \\(\\Delta\\) sMAPE \\\\\n"
)
latex += "\\midrule\n"

for model, row in table1.iterrows():
    latex += (
        f"{int(row['Rank'])} & {model} & "
        f"{row['MAE']:.2f} & {row['Delta_MAE']:+.2f} & "
        f"{row['RMSE']:.2f} & {row['Delta_RMSE']:+.2f} & "
        f"{row['MASE']:.2f} & {row['Delta_MASE']:+.2f} & "
        f"{row['sMAPE']:.2f} & {row['Delta_sMAPE']:+.2f} \\\\\n"
    )

latex += "\\bottomrule\n"
latex += "\\end{tabular}\n"
latex += (
    "\\caption{Overall model rankings based on mean MAE across all forecast horizons "
    "(clean weather). Delta columns show differences relative to the best MAE-ranked model.}\n"
)
latex += "\\label{tab:overall_rankings}\n"
latex += "\\end{table}\n"

with open("../results/tables/table1_overall_rankings.tex", "w") as f:
    f.write(latex)


In [197]:
# Figure 2: Performance by Horizon Line Plot
fig, ax = plt.subplots(figsize=(10, 6))

for i, model in enumerate(MODEL_ORDER):
    model_data = df_clean[df_clean['model'] == model].sort_values('horizon')
    ax.plot(
        model_data['horizon'], model_data['MAE_mean'],
        marker='o', label=model, color=colors[i % len(colors)]
    )

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE (bikes/hour)')
ax.set_ylim(120, 1200)
# move legend outside (right)
ax.legend(
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    framealpha=0.9,
    borderaxespad=0.0
)

ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xticks([6, 24, 48, 168])
ax.legend(
    loc='upper right',
    bbox_to_anchor=(0.93, 1.0),  # tweak these two numbers
    # bbox_to_anchor=(0.00, 1.0),  # tweak these two numbers
    framealpha=0.9,
    fontsize=9
)


# reserve space on the right for the legend
fig.subplots_adjust(right=0.78)

plt.savefig('../results/figures/fig2_performance_by_horizon.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig2_performance_by_horizon.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 2 saved")


Figure 2 saved


In [198]:
# Figure 14 (= 2 without TabPFN): Performance by Horizon Line Plot
fig, ax = plt.subplots(figsize=(10, 6))

for i, model in enumerate(MODEL_ORDER):
    if model != 'TabPFN':
        model_data = df_clean[df_clean['model'] == model].sort_values('horizon')
        ax.plot(
            model_data['horizon'], model_data['MAE_mean'],
            marker='o', label=model, color=colors[i % len(colors)]
        )

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE (bikes/hour)')
ax.set_ylim(120, 1200)


ax.legend(
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    framealpha=0.9,
    borderaxespad=0.0
)

ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xticks([6, 24, 48, 168])
ax.legend(
    loc='upper right',
    bbox_to_anchor=(0.93, 1.0),  # tweak these two numbers
    # bbox_to_anchor=(0.00, 1.0),  # tweak these two numbers
    framealpha=0.9,
    fontsize=9
)


# reserve space on the right for the legend
fig.subplots_adjust(right=0.78)

plt.savefig('../results/figures/fig14_performance_by_horizon_without_TabPFN.png', dpi=300, bbox_inches='tight')

plt.close()

print("Figure 14 saved")


Figure 14 saved


In [199]:
# Figure 3: TabPFN vs Baselines Grouped Bar Chart
baselines = ['SARIMAX', 'XGBoost', 'Seasonal_Naive']
models_to_compare = ['TabPFN'] + [m for m in baselines if m in df_clean['model'].unique()]

fig, ax = plt.subplots(figsize=(12, 6))

horizons = sorted(df_clean['horizon'].unique())
x = np.arange(len(horizons))
width = 0.2

for i, model in enumerate(models_to_compare):
    model_data = df_clean[df_clean['model'] == model].sort_values('horizon')
    offset = (i - len(models_to_compare)/2 + 0.5) * width
    ax.bar(x + offset, model_data['MAE_mean'], width, 
           label=model, color=colors[i % len(colors)])

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE (bikes/hour)')
ax.set_xticks(x)
ax.set_xticklabels(horizons)
ax.legend(loc='best', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../results/figures/fig3_tabpfn_vs_baselines.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig3_tabpfn_vs_baselines.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 3 saved")

Figure 3 saved


In [201]:
fig, ax = plt.subplots(figsize=(12, 6))

metrics = ["MAE", "RMSE", "MASE", "sMAPE"]

# numeric horizon order
horizons = sorted(df_clean["horizon"].astype(int).unique())
horizon_order = [f"{h}h" for h in horizons]

# build long rank table
rank_rows = []
for h in horizons:
    df_h = df_clean[df_clean["horizon"].astype(int) == h].copy()
    for m in metrics:
        col = f"{m}_mean"
        if col not in df_h.columns:
            continue
        df_h[f"{m}_rank"] = df_h[col].rank(method="min", ascending=True)
        rank_rows.append(
            df_h[["model", f"{m}_rank"]].assign(Horizon=f"{h}h", Metric=m)
            .rename(columns={f"{m}_rank": "Rank"})
        )

df_ranks = pd.concat(rank_rows, ignore_index=True)

# pivot to heatmap matrix: rows=models, cols=(Horizon, Metric)
df_pivot = df_ranks.pivot_table(
    index="model",
    columns=["Horizon", "Metric"],
    values="Rank"
)

df_pivot = df_pivot.reindex(MODEL_ORDER)
# enforce column order: horizon outer, metric inner
desired_cols = pd.MultiIndex.from_product(
    [horizon_order, metrics],
    names=["Horizon", "Metric"]
)
df_pivot = df_pivot.reindex(columns=desired_cols)

# plot
sns.heatmap(df_pivot, annot=True, fmt=".0f", cmap="RdYlGn_r", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Model")
plt.tight_layout()

plt.tight_layout()
plt.savefig('../results/figures/fig11_ranking_heatmap.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig11_ranking_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 11 saved")

Figure 11 saved


## Section 3: RQ1.2 Analysis - Weather Impact on TabPFN

In [204]:
# TabPFN vs TabPFN_NoWeather comparison
tabpfn_data = df_clean[df_clean['model'] == 'TabPFN'].sort_values('horizon')
tabpfn_noweather_data = df_clean[df_clean['model'] == 'TabPFN_NoWeather'].sort_values('horizon')

weather_benefit = pd.DataFrame({
    'Horizon': tabpfn_data['horizon'].values,
    'TabPFN': tabpfn_data['MAE_mean'].values,
    'TabPFN_NoWeather': tabpfn_noweather_data['MAE_mean'].values
})

weather_benefit['Improvement'] = (weather_benefit['TabPFN_NoWeather'] - weather_benefit['TabPFN']).round(2)
weather_benefit['Improvement_%'] = ((weather_benefit['Improvement'] / weather_benefit['TabPFN_NoWeather']) * 100).round(2)

weather_benefit = weather_benefit.round(2)

print("Weather Benefit Analysis:")
print(weather_benefit)

Weather Benefit Analysis:
   Horizon  TabPFN  TabPFN_NoWeather  Improvement  Improvement_%
0        6  170.50            441.42       270.92          61.37
1       24  169.57            266.10        96.53          36.28
2       48  208.94            298.72        89.78          30.06
3      168  176.70            317.73       141.03          44.39


In [205]:
# Export Table 2: TabPFN Weather Benefit
table2 = weather_benefit.copy()
table2.to_csv('../results/tables/table2_tabpfn_weather.csv', index=False)

# Generate LaTeX table
latex_table2 = "\\begin{table}[h]\n"
latex_table2 += "\\centering\n"
latex_table2 += "\\begin{tabular}{crrrr}\n"
latex_table2 += "\\toprule\n"
latex_table2 += "Horizon & TabPFN & TabPFN\\_NoWeather & Improvement & Improvement \\% \\\\\n"
latex_table2 += "\\midrule\n"

for _, row in table2.iterrows():
    latex_table2 += f"{int(row['Horizon'])} & {row['TabPFN']:.2f} & {row['TabPFN_NoWeather']:.2f} & {row['Improvement']:.2f} & {row['Improvement_%']:.2f} \\\\\n"

latex_table2 += "\\bottomrule\n"
latex_table2 += "\\end{tabular}\n"
latex_table2 += "\\caption{Impact of weather information on TabPFN performance across forecast horizons.}\n"
latex_table2 += "\\label{tab:tabpfn_weather}\n"
latex_table2 += "\\end{table}\n"

with open('../results/tables/table2_tabpfn_weather.tex', 'w') as f:
    f.write(latex_table2)

print("Table 2 exported")

Table 2 exported


In [206]:
# Figure 4: TabPFN Weather Benefit Side-by-Side Bars
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(weather_benefit))
width = 0.35

bars1 = ax.bar(x - width/2, weather_benefit['TabPFN'], width, 
               label='TabPFN (with weather)', color=colors[0])
bars2 = ax.bar(x + width/2, weather_benefit['TabPFN_NoWeather'], width, 
               label='TabPFN (no weather)', color=colors[1])

# Add percentage improvement annotations
for i, (bar1, bar2, pct) in enumerate(zip(bars1, bars2, weather_benefit['Improvement_%'])):
    height = max(bar1.get_height(), bar2.get_height())
    ax.text(i, height + 0.5, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE (bikes/hour)')
ax.set_xticks(x)
ax.set_xticklabels(weather_benefit['Horizon'])
ax.legend(loc='best', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../results/figures/fig4_tabpfn_weather_benefit.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig4_tabpfn_weather_benefit.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 4 saved")

Figure 4 saved


In [207]:
# Figure 5: Weather Benefit by Horizon Line Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(weather_benefit['Horizon'], weather_benefit['Improvement'], 
        marker='o', color=colors[2], linewidth=2, markersize=8)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE Improvement (bikes/hour)')
ax.set_xticks(weather_benefit['Horizon'])
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../results/figures/fig5_weather_benefit_by_horizon.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig5_weather_benefit_by_horizon.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 5 saved")

Figure 5 saved


## Section 4: RQ1.3 Analysis - Forecast Uncertainty Impact

In [208]:
# Models available in both scenarios
models_both_scenarios = ['SARIMAX', 'TabPFN', 'XGBoost']

# Overall degradation summary
degradation_summary = []

for model in models_both_scenarios:
    clean_mae = df_clean[df_clean['model'] == model]['MAE_mean'].mean()
    degraded_mae = df_degraded[df_degraded['model'] == model]['MAE_mean'].mean()
    increase = degraded_mae - clean_mae
    increase_pct = (increase / clean_mae) * 100
    
    degradation_summary.append({
        'Model': model,
        'Clean_MAE': round(clean_mae, 2),
        'Degraded_MAE': round(degraded_mae, 2),
        'Increase': round(increase, 2),
        'Increase_%': round(increase_pct, 2)
    })

degradation_df = pd.DataFrame(degradation_summary).sort_values('Increase_%')

print("Degradation Summary:")
print(degradation_df)

Degradation Summary:
     Model  Clean_MAE  Degraded_MAE  Increase  Increase_%
0  SARIMAX     304.32        307.87      3.54        1.16
2  XGBoost     880.20        972.93     92.73       10.53
1   TabPFN     181.43        216.00     34.57       19.05


In [209]:
# Degradation by horizon
degradation_by_horizon = []

horizons = sorted(df_clean['horizon'].unique())

for horizon in horizons:
    row_data = {'Horizon': horizon}
    
    for model in models_both_scenarios:
        clean_mae = df_clean[(df_clean['model'] == model) & (df_clean['horizon'] == horizon)]['MAE_mean'].values[0]
        degraded_mae = df_degraded[(df_degraded['model'] == model) & (df_degraded['horizon'] == horizon)]['MAE_mean'].values[0]
        increase_pct = ((degraded_mae - clean_mae) / clean_mae) * 100
        row_data[f'{model}_%'] = round(increase_pct, 2)
    
    degradation_by_horizon.append(row_data)

degradation_horizon_df = pd.DataFrame(degradation_by_horizon)

print("\nDegradation by Horizon:")
print(degradation_horizon_df)


Degradation by Horizon:
   Horizon  SARIMAX_%  TabPFN_%  XGBoost_%
0        6       1.77     14.56      -1.38
1       24       2.54     13.45      -2.98
2       48       2.47     13.09       9.53
3      168      -1.82     35.82      28.25


In [210]:
# Correlation between horizon and degradation
print("\nCorrelation between horizon and degradation:")
for model in models_both_scenarios:
    col = f'{model}_%'
    corr = degradation_horizon_df['Horizon'].corr(degradation_horizon_df[col])
    print(f"{model}: {corr:.3f}")


Correlation between horizon and degradation:
SARIMAX: -0.928
TabPFN: 0.958
XGBoost: 0.974


In [211]:
# Export Table 3: Degradation Summary
table3 = degradation_df.copy()
table3.to_csv('../results/tables/table3_degradation_summary.csv', index=False)

# Generate LaTeX table
latex_table3 = "\\begin{table}[h]\n"
latex_table3 += "\\centering\n"
latex_table3 += "\\begin{tabular}{lrrrr}\n"
latex_table3 += "\\toprule\n"
latex_table3 += "Model & Clean MAE & Degraded MAE & Increase & Increase \\% \\\\\n"
latex_table3 += "\\midrule\n"

for _, row in table3.iterrows():
    latex_table3 += f"{row['Model']} & {row['Clean_MAE']:.2f} & {row['Degraded_MAE']:.2f} & {row['Increase']:.2f} & {row['Increase_%']:.2f} \\\\\n"

latex_table3 += "\\bottomrule\n"
latex_table3 += "\\end{tabular}\n"
latex_table3 += "\\caption{Performance degradation from clean to degraded weather forecasts (averaged across all horizons).}\n"
latex_table3 += "\\label{tab:degradation_summary}\n"
latex_table3 += "\\end{table}\n"

with open('../results/tables/table3_degradation_summary.tex', 'w') as f:
    f.write(latex_table3)

print("Table 3 exported")

Table 3 exported


In [212]:
# Export Table 4: Degradation by Horizon
table4 = degradation_horizon_df.copy()
table4.to_csv('../results/tables/table4_degradation_by_horizon.csv', index=False)

# Generate LaTeX table
latex_table4 = "\\begin{table}[h]\n"
latex_table4 += "\\centering\n"
latex_table4 += "\\begin{tabular}{c" + "r" * len(models_both_scenarios) + "}\n"
latex_table4 += "\\toprule\n"
latex_table4 += "Horizon & " + " & ".join([f"{m} \\%" for m in models_both_scenarios]) + " \\\\\n"
latex_table4 += "\\midrule\n"

for _, row in table4.iterrows():
    latex_table4 += f"{int(row['Horizon'])}"
    for model in models_both_scenarios:
        latex_table4 += f" & {row[f'{model}_%']:.2f}"
    latex_table4 += " \\\\\n"

latex_table4 += "\\bottomrule\n"
latex_table4 += "\\end{tabular}\n"
latex_table4 += "\\caption{Performance degradation percentage by forecast horizon.}\n"
latex_table4 += "\\label{tab:degradation_by_horizon}\n"
latex_table4 += "\\end{table}\n"

with open('../results/tables/table4_degradation_by_horizon.tex', 'w') as f:
    f.write(latex_table4)

print("Table 4 exported")

Table 4 exported


In [213]:
# Figure 6: Degradation Overview Grouped Bars
fig, ax = plt.subplots(figsize=(10, 6))

models_both_scenarios = [m for m in MODEL_ORDER if m in degradation_df['Model'].values]

x = np.arange(len(models_both_scenarios))
width = 0.35

clean_values = [degradation_df[degradation_df['Model'] == m]['Clean_MAE'].values[0] for m in models_both_scenarios]
degraded_values = [degradation_df[degradation_df['Model'] == m]['Degraded_MAE'].values[0] for m in models_both_scenarios]
pct_increases = [degradation_df[degradation_df['Model'] == m]['Increase_%'].values[0] for m in models_both_scenarios]

bars1 = ax.bar(x - width/2, clean_values, width, label='Clean weather', color=colors[0])
bars2 = ax.bar(x + width/2, degraded_values, width, label='Degraded weather', color=colors[3])



# Add percentage increase annotations
for i, (bar2, pct) in enumerate(zip(bars2, pct_increases)):
    height = bar2.get_height()
    ax.text(i + width/2, height + 0.5, f'+{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model')
ax.set_ylabel('MAE (bikes/hour)')
ax.set_xticks(x)
ax.set_xticklabels(models_both_scenarios)
ax.legend(loc='best', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../results/figures/fig6_degradation_overview.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig6_degradation_overview.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 6 saved")

Figure 6 saved


In [214]:
# Figure 7: Degradation by Horizon Line Plot
fig, ax = plt.subplots(figsize=(10, 6))

for i, model in enumerate(models_both_scenarios):
    col = f'{model}_%'
    ax.plot(degradation_horizon_df['Horizon'], degradation_horizon_df[col], 
            marker='o', label=model, color=colors[i % len(colors)])

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('Performance Degradation (%)')
ax.legend(loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xticks(degradation_horizon_df['Horizon'])
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/figures/fig7_degradation_by_horizon.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig7_degradation_by_horizon.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 7 saved")

Figure 7 saved


In [215]:
# Figure 8: Robustness Ranking Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

robustness_data = degradation_df.sort_values('Increase_%', ascending=True)
y_pos = np.arange(len(robustness_data))

bars = ax.barh(y_pos, robustness_data['Increase_%'], color=colors[:len(robustness_data)])

ax.set_yticks(y_pos)
ax.set_yticklabels(robustness_data['Model'])
ax.set_xlabel('Average Degradation (%)')
ax.set_ylabel('Model')
ax.grid(axis='x', alpha=0.3, linestyle='--')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, robustness_data['Increase_%'])):
    ax.text(val + 0.2, i, f'{val:.2f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/fig8_robustness_ranking.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig8_robustness_ranking.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 8 saved")

Figure 8 saved


In [216]:
# Figure 9: Degradation Heatmap
fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data for heatmap
heatmap_data = degradation_horizon_df.set_index('Horizon')[[f'{m}_%' for m in models_both_scenarios]]
heatmap_data.columns = models_both_scenarios
heatmap_data = heatmap_data.T

# Create heatmap
im = ax.imshow(heatmap_data.values, cmap='Reds', aspect='auto')

# Set ticks and labels
ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_xticklabels(heatmap_data.columns)
ax.set_yticklabels(heatmap_data.index)

ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('Model')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('MAE Increase (%)', rotation=270, labelpad=20)

# Add text annotations
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        text = ax.text(j, i, f"{heatmap_data.values[i, j]:.1f}",
                      ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/fig9_degradation_heatmap.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig9_degradation_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 9 saved")

Figure 9 saved


## Section 5: Variance Analysis

In [217]:
# Calculate CV (coefficient of variation) across folds
stability_metrics = []

for scenario in ['clean_only', 'degraded']:
    scenario_data = df_detailed_latest[df_detailed_latest['weather_scenario'] == scenario]
    
    for model in scenario_data['model'].unique():
        model_data = scenario_data[scenario_data['model'] == model]
        
        mean_mae = model_data.groupby('fold')['MAE'].mean().mean()
        std_mae = model_data.groupby('fold')['MAE'].mean().std()
        cv = (std_mae / mean_mae) * 100 if mean_mae > 0 else 0
        
        stability_metrics.append({
            'Model': model,
            'Scenario': scenario,
            'Mean_MAE': round(mean_mae, 2),
            'Std_MAE': round(std_mae, 2),
            'CV_%': round(cv, 2)
        })

stability_df = pd.DataFrame(stability_metrics).sort_values(['Scenario', 'Model'])

print("Stability Metrics:")
print(stability_df)

Stability Metrics:
              Model    Scenario  Mean_MAE  Std_MAE   CV_%
1             ARIMA  clean_only    728.29   128.13  17.59
2           SARIMAX  clean_only    304.32    52.59  17.28
0    Seasonal_Naive  clean_only    396.97   147.96  37.27
3            TabPFN  clean_only    181.43    40.32  22.22
4  TabPFN_NoWeather  clean_only    330.99   113.72  34.36
5           XGBoost  clean_only    880.20   138.06  15.69
6           SARIMAX    degraded    307.87    51.79  16.82
7            TabPFN    degraded    216.00    46.01  21.30
8           XGBoost    degraded    972.93   173.59  17.84


In [218]:
# Export Table 5: Stability Metrics
table5 = stability_df.copy()
table5.to_csv('../results/tables/table5_stability.csv', index=False)

# Generate LaTeX table
latex_table5 = "\\begin{table}[h]\n"
latex_table5 += "\\centering\n"
latex_table5 += "\\begin{tabular}{llrrr}\n"
latex_table5 += "\\toprule\n"
latex_table5 += "Model & Scenario & Mean MAE & Std MAE & CV \\% \\\\\n"
latex_table5 += "\\midrule\n"

for _, row in table5.iterrows():
    scenario = row["Scenario"].replace("_", r"\_")
    latex_table5 += f"{row['Model']} & {scenario} & {row['Mean_MAE']:.2f} & {row['Std_MAE']:.2f} & {row['CV_%']:.2f} \\\\\n"


latex_table5 += "\\bottomrule\n"
latex_table5 += "\\end{tabular}\n"
latex_table5 += "\\caption{Model stability metrics across cross-validation folds.}\n"
latex_table5 += "\\label{tab:stability}\n"
latex_table5 += "\\end{table}\n"

with open('../results/tables/table5_stability.tex', 'w') as f:
    f.write(latex_table5)

print("Table 5 exported")

Table 5 exported


In [219]:
# Figure 10: Variance Comparison Box Plots
# Select models that exist in both scenarios for fair comparison
models_in_both = set(df_detailed_clean['model'].unique()) & set(df_detailed_degraded['model'].unique())
models_to_plot = sorted(list(models_in_both))

n_models = len(models_to_plot)
n_cols = min(3, n_models)
n_rows = int(np.ceil(n_models / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
if n_models == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, model in enumerate(models_to_plot):
    ax = axes[idx]
    
    # Get fold-level MAE for this model
    clean_data = df_detailed_clean[df_detailed_clean['model'] == model].groupby('fold')['MAE'].mean()
    degraded_data = df_detailed_degraded[df_detailed_degraded['model'] == model].groupby('fold')['MAE'].mean()
    
    data_to_plot = [clean_data.values, degraded_data.values]
    
    bp = ax.boxplot(data_to_plot, labels=['Clean', 'Degraded'], patch_artist=True)
    
    # Color boxes
    for patch, color in zip(bp['boxes'], [colors[0], colors[3]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_ylabel('MAE (bikes/hour)')
    ax.set_title(model, fontsize=10, pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

# Hide empty subplots
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('../results/figures/fig10_variance_comparison.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig10_variance_comparison.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 10 saved")

Figure 10 saved


In [220]:
print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print("\nGenerated outputs:")
print("- 10 figures (PNG + PDF)")
print("- 5 tables (CSV + TEX)")
print("\nAll files saved to results/ directory")


ANALYSIS COMPLETE

Generated outputs:
- 10 figures (PNG + PDF)
- 5 tables (CSV + TEX)

All files saved to results/ directory


In [221]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

fig, (ax_hm, ax_ln) = plt.subplots(
    1, 2,
    figsize=(12, 4.2),                  # wider overall figure
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.0, 1.35]}  # give more width to line plot
)

# ---------- Heatmap ----------
metrics = ["MAE", "RMSE", "MASE", "sMAPE"]
horizons = sorted(df_clean["horizon"].astype(int).unique())
horizon_order = [f"{h}h" for h in horizons]

rank_rows = []
for h in horizons:
    df_h = df_clean[df_clean["horizon"].astype(int) == h].copy()
    for m in metrics:
        col = f"{m}_mean"
        if col not in df_h.columns:
            continue
        df_h[f"{m}_rank"] = df_h[col].rank(method="min", ascending=True)
        rank_rows.append(
            df_h[["model", f"{m}_rank"]]
            .assign(Horizon=f"{h}h", Metric=m)
            .rename(columns={f"{m}_rank": "Rank"})
        )

df_ranks = pd.concat(rank_rows, ignore_index=True)
df_pivot = df_ranks.pivot_table(index="model", columns=["Horizon", "Metric"], values="Rank")
df_pivot = df_pivot.reindex(MODEL_ORDER)

desired_cols = pd.MultiIndex.from_product([horizon_order, metrics], names=["Horizon", "Metric"])
df_pivot = df_pivot.reindex(columns=desired_cols)
df_pivot.index.name = None
sns.heatmap(
    df_pivot,
    annot=True, fmt=".0f",
    cmap="RdYlGn_r",
    ax=ax_hm,
    cbar=False,
    annot_kws={"fontsize": 7}
)

ax_hm.set_xlabel("")
# ax_hm.set_ylabel("Model")
ax_hm.tick_params(axis="x", labelrotation=90, labelsize=7)
ax_hm.tick_params(axis="y", labelsize=8)


# ---------- Line plot ----------
for i, model in enumerate(MODEL_ORDER):
    model_data = df_clean[df_clean["model"] == model].sort_values("horizon")
    ax_ln.plot(
        model_data["horizon"].astype(int),
        model_data["MAE_mean"],
        marker="o",
        label=model,
        color=colors[i % len(colors)]
    )

ax_ln.set_xlabel("Horizon (h)")
ax_ln.set_ylabel("MAE")
ax_ln.set_xticks([6, 24, 48, 168])
ax_ln.grid(True, alpha=0.3, linestyle="--")

ax_ln.legend(
    loc="upper right",
    bbox_to_anchor=(0.94, 1.0),  # small inward shift (x, y)
    framealpha=0.9,
    fontsize=8
)


out_png = "../results/figures/fig12_heatmap_MAE_by_horizon.png"
# out_pdf = "../results/figures/fig12_heatmap_MAE_by_horizon.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
# fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.close(fig)


In [222]:
# Figure 13: MAE Distribution Across Models and Horizons

horizons = sorted(df_detailed_clean['horizon'].unique())
n_horizons = len(horizons)

fig, axes = plt.subplots(1, n_horizons, figsize=(5*n_horizons, 5))
if n_horizons == 1:
    axes = [axes]

model_colors = {
    'Seasonal_Naive': colors[0],
    'ARIMA': colors[1],
    'SARIMAX': colors[2],
    'XGBoost': colors[3],
    'TabPFN_NoWeather': colors[4],
    'TabPFN': colors[5]
}

# Calculate global y-axis limits for consistent scale
all_mae_values = df_detailed_clean['MAE'].values
y_min = np.floor(all_mae_values.min())
# y_max = np.ceil(all_mae_values.max())
y_max = 2000

for col_idx, horizon in enumerate(horizons):
    df_h = df_detailed_clean[df_detailed_clean['horizon'] == horizon].copy()
    
    ax = axes[col_idx]
    available_models = [m for m in MODEL_ORDER if m in df_h['model'].unique()]
    data_mae = [df_h[df_h['model'] == m]['MAE'].values for m in available_models]
    colors_mae = [model_colors.get(m, colors[6]) for m in available_models]
    
    bp = ax.boxplot(data_mae, tick_labels=available_models, patch_artist=True,
                    widths=0.6, showfliers=False)
    for patch, color in zip(bp['boxes'], colors_mae):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(f'{horizon}h Forecast', fontweight='bold', fontsize=10)
    ax.set_ylabel('MAE (bikes/hour)', fontsize=10)
    ax.set_xticklabels(available_models, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(y_min, y_max)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('../results/figures/fig13_error_distributions.png', dpi=300, bbox_inches='tight')
# plt.savefig('../results/figures/fig13_error_distributions.pdf', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 13 saved")
# ```

# **Caption:**
# ```
# Figure 13: MAE distributions across models and forecast horizons. Box plots show the distribution of mean absolute errors from cross-validation folds for each model at each forecast horizon (1, 6, 12, 18, 24 hours). All subplots share the same y-axis scale to enable fair comparison of variance across horizons. Models are ordered by performance, with narrower boxes indicating more consistent predictions across folds.

Figure 13 saved
